# Feature Scoring — Input × Outcome Importance Matrix

Uses `ema_workbench.analysis.feature_scoring` (Extra Trees) to
compute how much each uncertainty / lever drives each outcome.
Produces a heatmap: inputs on one axis, outcomes on the other,
cells coloured by importance score.

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
FOLDER_NAME = FOLDER_NAME

# Outcomes to include in the heatmap (set to None for all scalar outcomes)
SELECTED_OUTCOMES = [
    "death_rate",
    "safe_rate",
    "death_rate_T1_BCAS",
    "death_rate_T2_BCAS",
    "death_rate_T3_BCAS",
    "mean_wait_care_T1_BCAS",
    "mean_wait_care_T2_BCAS",
    "mean_wait_care_T3_BCAS",
    "mean_wait_transport_T1_BCAS",
    "mean_wait_transport_T2_BCAS",
    "mean_wait_transport_T3_BCAS",
    "total_transport_time",
]


In [ ]:
import pandas as pd
import numpy as np

run_folder = paths.MULTI_RUN_OUTPUT / FOLDER_NAME
df = pd.read_csv(run_folder / "results.csv")
print(f"Loaded {len(df)} runs  |  {df['scenario'].nunique()} scenarios  |  "
      f"{df['policy'].nunique()} distinct policies")

In [ ]:
# Columns before 'scenario' are inputs; after 'model' are outcomes.
meta_cols  = ["scenario", "policy", "model"]
meta_start = min(df.columns.get_loc(c) for c in meta_cols if c in df.columns)
input_cols = df.columns[:meta_start].tolist()
model_idx  = df.columns.get_loc("model")
outcome_cols = df.columns[model_idx + 1:].tolist()

print(f"Inputs  ({len(input_cols)}): {input_cols}")
print(f"Outcomes ({len(outcome_cols)}): {len(outcome_cols)} total")

if SELECTED_OUTCOMES is not None:
    outcome_cols = [c for c in SELECTED_OUTCOMES if c in outcome_cols]
    print(f"Selected {len(outcome_cols)} outcomes for scoring")

In [ ]:
# x = inputs (uncertainties + levers), y = dict of outcome arrays
# Drop rows with NaN in any selected outcome
keep_cols = input_cols + outcome_cols
df_clean = df[keep_cols].dropna()
print(f"Kept {len(df_clean)}/{len(df)} rows after dropping NaN")

x = df_clean[input_cols].astype(float)
y = {col: df_clean[col].values for col in outcome_cols}

In [ ]:
from ema_workbench.analysis import feature_scoring

scores = feature_scoring.get_feature_scores_all(x, y)
print(f"Score matrix shape: {scores.shape}")
print(f"Inputs: {list(scores.index)}")
print(f"Outcomes: {list(scores.columns)}")
scores

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(max(12, len(outcome_cols) * 1.2),
                                max(6, len(input_cols) * 0.5)))
sns.heatmap(
    scores,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    vmin=0,
    vmax=1,
)
ax.set_title("Feature Scoring — Input Importance per Outcome", fontsize=14)
ax.set_xlabel("Outcome")
ax.set_ylabel("Input (uncertainty / lever)")
plt.tight_layout()
plt.show()

## Top inputs per outcome

For each outcome, list the inputs ranked by importance score.

In [ ]:
TOP_N = 5

for outcome in scores.columns:
    ranked = scores[outcome].sort_values(ascending=False)
    print(f"\n{outcome}:")
    for i, (inp, sc) in enumerate(ranked.head(TOP_N).items()):
        bar = '█' * int(sc * 40)
        print(f"  {i+1}. {inp:35s} {sc:.3f}  {bar}")

## Per-Policy Feature Scoring

Run feature scoring separately for each policy (lever combination)
to see whether input importance shifts across policies.

In [ ]:
lever_cols = [c for c in input_cols if c in ["facility_eligibility", "ambulance_eligibility"]]
uncertainty_cols = [c for c in input_cols if c not in lever_cols]

policies = df[lever_cols].drop_duplicates().sort_values(lever_cols).reset_index(drop=True)
print(f"{len(policies)} distinct policies:")
display(policies)

for idx, policy_row in policies.iterrows():
    mask = pd.Series(True, index=df.index)
    for col in lever_cols:
        mask &= df[col] == policy_row[col]
    df_pol = df.loc[mask, uncertainty_cols + outcome_cols].dropna()
    
    x_pol = df_pol[uncertainty_cols].astype(float)
    y_pol = {col: df_pol[col].values for col in outcome_cols}
    
    scores_pol = feature_scoring.get_feature_scores_all(x_pol, y_pol)
    
    label = ", ".join(f"{c}={int(policy_row[c])}" for c in lever_cols)
    fig, ax = plt.subplots(figsize=(max(12, len(outcome_cols) * 1.2),
                                    max(6, len(uncertainty_cols) * 0.5)))
    sns.heatmap(
        scores_pol,
        annot=True,
        fmt=".2f",
        cmap="YlOrRd",
        linewidths=0.5,
        ax=ax,
        vmin=0,
        vmax=1,
    )
    ax.set_title(f"Feature Scoring — {label}", fontsize=14)
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Uncertainty")
    plt.tight_layout()
    plt.show()